In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from datetime import datetime
import json

In [2]:
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

IMAGE_SIZE_TL = (96, 96)
BATCH_SIZE = 32

In [3]:

train_df = pd.read_csv("../data/processed/train_split.csv")
val_df   = pd.read_csv("../data/processed/val_split.csv")
test_df  = pd.read_csv("../data/processed/test_split.csv")

with open("../artifacts/class_mapping.json", "r") as f:
    mapping = json.load(f)

classes = mapping["classes"]
num_classes = len(classes)
print("Classes:", num_classes)

Classes: 36


In [4]:
def load_image_rgb(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=1)
    img = tf.image.resize(img, IMAGE_SIZE_TL)
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.image.grayscale_to_rgb(img)  # (H,W,3)
    return img, label

def make_dataset(df, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label_id"].values))
    ds = ds.map(load_image_rgb, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000, seed=SEED)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df, shuffle=False)
test_ds  = make_dataset(test_df, shuffle=False)


MobileNetV2 model

In [5]:
base = tf.keras.applications.MobileNetV2(
    input_shape=(IMAGE_SIZE_TL[0], IMAGE_SIZE_TL[1], 3),
    include_top=False,
    weights="imagenet"
)
base.trainable = False  # baseline TL

inputs = tf.keras.Input(shape=(IMAGE_SIZE_TL[0], IMAGE_SIZE_TL[1], 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model3 = tf.keras.Model(inputs, outputs)
model3.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply (Multiply)             │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 36)             │        46,116 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,304,100 (8.79 MB)

 Trainable params: 46,116 (180.14 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
model3.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5),
]

history3 = model3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=6,
    callbacks=callbacks
)


Epoch 1/6
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 1454s 446ms/step - accuracy: 0.9721 - loss: 0.1144 - val_accuracy: 0.9958 - val_loss: 0.0166 - learning_rate: 0.0010
Epoch 2/6
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 1451s 462ms/step - accuracy: 0.9956 - loss: 0.0167 - val_accuracy: 0.9962 - val_loss: 0.0142 - learning_rate: 0.0010
Epoch 3/6
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 1755s 563ms/step - accuracy: 0.9969 - loss: 0.0109 - val_accuracy: 0.9961 - val_loss: 0.0121 - learning_rate: 0.0010
Epoch 4/6
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 1460s 468ms/step - accuracy: 0.9973 - loss: 0.0082 - val_accuracy: 0.9982 - val_loss: 0.0068 - learning_rate: 0.0010
Epoch 5/6
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 1297s 416ms/step - accuracy: 0.9975 - loss: 0.0077 - val_accuracy: 0.9988 - val_loss: 0.0047 - learning_rate: 0.0010
Epoch 6/6
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 733s 235ms/step - accuracy: 0.9983 - loss: 0.0055 - val_accuracy: 0.9980 - val_loss: 0.0117 - learning_rate: 0.0010


In [7]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model3.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

history3_ft = model3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=1, restore_best_weights=True)]
)


Epoch 1/3
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 832s 264ms/step - accuracy: 0.9871 - loss: 0.0528 - val_accuracy: 0.9943 - val_loss: 0.0262
Epoch 2/3
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 821s 264ms/step - accuracy: 0.9955 - loss: 0.0196 - val_accuracy: 0.9982 - val_loss: 0.0059
Epoch 3/3
3112/3112 ━━━━━━━━━━━━━━━━━━━━ 932s 299ms/step - accuracy: 0.9963 - loss: 0.0141 - val_accuracy: 0.9985 - val_loss: 0.0046


In [8]:
test_loss3, test_acc3 = model3.evaluate(test_ds)
print("Model 3 Test Accuracy:", test_acc3)


MODELS_DIR = Path("../models"); MODELS_DIR.mkdir(exist_ok=True)
ART_DIR = Path("../artifacts"); ART_DIR.mkdir(exist_ok=True)

run_id3 = datetime.now().strftime("%Y%m%d_%H%M%S")
model3.save(MODELS_DIR / "model3_mobilenetv2.keras")

pd.DataFrame(history3.history).assign(model_version="model3_mobilenetv2", run_id=run_id3)\
  .to_csv(ART_DIR / f"train_history_model3_{run_id3}.csv", index=False)

model_card3 = {
    "model_version": "model3_mobilenetv2",
    "image_size": list(IMAGE_SIZE_TL),
    "notes": "Transfer learning MobileNetV2 (ImageNet), grayscale->RGB, optional fine-tune."
}
with open(ART_DIR / f"model_card_model3_{run_id3}.json", "w") as f:
    json.dump(model_card3, f, indent=2)

print(" Saved Model 3 + artifacts")


667/667 ━━━━━━━━━━━━━━━━━━━━ 157s 234ms/step - accuracy: 0.9988 - loss: 0.0044
Model 3 Test Accuracy: 0.9987816214561462
 Saved Model 3 + artifacts
